# Task 6 · Inference and scoring only
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
import codecs

def stream_run(cmd, check=True, cwd=None):
    # Route child output through IPython stdout, rather than its hidden OS file descriptor.
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                               cwd=cwd if cwd is not None else globals().get('PROJECT_ROOT'))
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True))
        sys.stdout.flush()
        result = subprocess.CompletedProcess(cmd, process.wait())
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if check:
        result.check_returncode()
    return result

checkout = Path(CHECKOUT).expanduser()
if not checkout.is_absolute():
    raise ValueError('CHECKOUT must be absolute, e.g. /content/semeval')
checkout = checkout.resolve()
if not checkout.exists():
    checkout.parent.mkdir(parents=True, exist_ok=True)
    stream_run(['git', 'clone', REPO_URL, str(checkout)], check=True, cwd=checkout.parent)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
stream_run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True, cwd=checkout.parent)
stream_run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True, cwd=checkout.parent)
PROJECT_ROOT = (checkout / PROJECT_SUBDIR).resolve()
SRC = PROJECT_ROOT / 'src'
if not (SRC / 'mmcultureqa/__init__.py').is_file():
    raise FileNotFoundError(f'Project package not found: {SRC / "mmcultureqa"}')
os.chdir(PROJECT_ROOT)
# Editable installation alone does not refresh sys.path in an already running kernel.
# Export PYTHONPATH too, so child Python processes and workers use this same checkout.
sys.path[:] = [str(SRC)] + [p for p in sys.path if p != str(SRC)]
python_paths = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p and p != str(SRC)]
os.environ['PYTHONPATH'] = os.pathsep.join([str(SRC), *python_paths])
import importlib
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == 'mmcultureqa' or name.startswith('mmcultureqa.'):
        del sys.modules[name]
stream_run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(PROJECT_ROOT)], check=True)
from mmcultureqa.common import ROOT as PACKAGE_ROOT
if PACKAGE_ROOT.resolve() != PROJECT_ROOT:
    raise ImportError(f'mmcultureqa loaded from {PACKAGE_ROOT}, expected {PROJECT_ROOT}')
stream_run([sys.executable, '-c', 'import mmcultureqa; print("Child package:", mmcultureqa.__file__)'])
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
INSTALL_VLLM = True  # final generation uses vLLM after training exits
requirements = 'requirements-vllm.txt' if INSTALL_VLLM else 'requirements.txt'
stream_run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / requirements)])
# Colab may have unrelated preinstalled-package conflicts; print the full report.
stream_run([sys.executable, '-m', 'pip', 'check'], check=False)
stream_run(['nvidia-smi'])


In [ ]:
# Fail early on dependency/import errors before downloading data or starting training.
import subprocess, sys
stream_run([sys.executable, '-c', """
import sys
from importlib.metadata import version
for name in ('numpy', 'torch', 'transformers', 'peft', 'trl', 'ms-swift', 'bitsandbytes', 'msgspec'):
    print(name, version(name), flush=True)
from transformers.modeling_utils import PreTrainedModel
from peft import PeftModel
from swift.llm import TrainArguments, RLHFArguments
import qwen_vl_utils
import decord
from bitsandbytes.nn import Linear4bit
from swift.llm import PtEngine
import mmcultureqa.common
import mmcultureqa.data.prepare
import mmcultureqa.training.sft
import mmcultureqa.training.grpo
import mmcultureqa.training.curves
import mmcultureqa.inference.predict
import mmcultureqa.evaluate
from swift.llm.train.sft import SwiftSft
from swift.llm.train.rlhf import SwiftRLHF
# Import the concrete trainer too: SwiftRLHF alone leaves its dependencies lazy.
from swift.trainers.rlhf_trainer.grpo_trainer import GRPOTrainer
from msgspec import field
from swift.llm import InferRequest, RequestConfig
if sys.argv[1] == '1':
    print('vllm', version('vllm'), flush=True)
    from vllm import LLM
    from swift.llm import VllmEngine
from swift.plugin import metric_mapping, orms, ORM
from swift.utils import Serializer
print('Training imports OK', flush=True)
""", '1' if INSTALL_VLLM else '0'])


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = '1kcdByZiSJzGSoEtY9q0OEhsZwiTZkxpV'  # ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = 'iamtro'  # Google Drive remote in rclone.conf


In [ ]:
stream_run(['sudo', 'apt-get', '-qq', 'update'])
stream_run(['sudo', 'apt-get', '-qq', 'install', '-y', 'rclone'])
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_absolute():
    raise ValueError('RCLONE_CONFIG_PATH must be absolute, e.g. /content/rclone.conf')
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    stream_run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
stream_run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), PROJECT_ROOT / 'data/raw')
else:
    stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Experiment 03 settings
SFT trains BF16 LoRA with microbatch 8 and accumulation 1.
Validation logs loss/token accuracy and vLLM-generated F1; F1 selects the best checkpoint.
Training model and optimizer move to RAM while generation and scoring run sequentially on GPU.
Each eval recreates vLLM; this is process isolation, not a persistent sleep-mode engine.
BF16 8B training may require more VRAM than an L4; sufficient host RAM is also required.
Default 1 epoch, peak learning rate 5e-5 (3% warmup), eval/save every 170 optimizer steps.
After training exits, merge the selected local adapter into a fresh BF16 base and upload best_model/.
Final validation and test inference use that complete BF16 model via vLLM in a separate process.
Drive uploads exclude local adapters, intermediate checkpoints, optimizer and temporary files.
Use the same ROOT_FOLDER_ID and RUN_NAME for SFT and inference. GRPO is unchanged and deferred.


In [ ]:
import json
USE_4BIT = False
PREDICT_WITH_GENERATE = False  # SFT generation + mmBERT; False selects teacher-forced validation
RUN_FULL = True  # full training from YAML; False explicitly selects a one-step smoke run
RUN_NAME = 'experiment03'
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME must contain only letters, numbers, underscores or hyphens')
RUNS = PROJECT_ROOT / 'outputs/runs' / RUN_NAME
RUNS.mkdir(parents=True, exist_ok=True)
SFT_STAGE = 'sft' if RUN_FULL else 'sft-smoke'
GRPO_STAGE = 'grpo' if RUN_FULL else 'grpo-smoke'
SFT = str(RUNS / SFT_STAGE)
GRPO = str(RUNS / GRPO_STAGE)
common = {'model.dtype': 'bfloat16', 'model.load_in_4bit': USE_4BIT,
          'training.predict_with_generate': PREDICT_WITH_GENERATE}

def upload_run(local_output, cloud_subdir):
    from mmcultureqa.storage import upload_filters
    copy(local_output, cloud(cloud_subdir), *upload_filters('merged_run'))

def upload_predictions(local_output, cloud_subdir):
    from mmcultureqa.storage import upload_filters
    copy(local_output, cloud(cloud_subdir), *upload_filters('predictions'))

def run_command(module, config, overrides):
    config = Path(config)
    if not config.is_absolute():
        config = PROJECT_ROOT / config
    if not config.is_file():
        raise FileNotFoundError(f'Configuration not found: {config}')
    cmd = [sys.executable, '-m', module, '--config', str(config)]
    for key, value in overrides.items():
        cmd += ['--set', f'{key}={json.dumps(value)}']
    stream_run(cmd, check=True)

def require_adapter(folder):
    folder = Path(folder)
    if not (folder / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Adapter config missing: {folder}')
    if not any((folder / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')):
        raise FileNotFoundError(f'Adapter weights missing: {folder}')


## Test inference from the merged BF16 model
Download the selected run's best_model from Drive. No original Qwen/adapter download is needed.
Model and processor loading are local-files-only; missing files raise an error instead of falling back to the Hub.
Uses the saved prompt and preprocessing from the exported training run.
Supports merged Qwen3-VL-4B and Qwen3-VL-8B checkpoints.
Set INPUT_RUN_NAME to the training run to download; model family and prompt restore from that artifact.
Validation scoring belongs to the SFT/GRPO notebooks. This notebook accepts test splits only.
Current devtest is unlabelled: generate predictions to submit; no local score can be computed.


In [ ]:
INPUT_RUN_NAME = RUN_NAME  # experiment03 / experiment05_qwen3vl8b
MODEL_STAGE = SFT_STAGE
SPLIT = 'devtest'
TEST_LIMIT = None  # full split; optionally 4 for an inference smoke test
if SPLIT not in {'devtest', 'test'}:
    raise ValueError('Use devtest/test here; validation is evaluated in the training notebooks')
if not re.fullmatch(r'[A-Za-z0-9_-]+', INPUT_RUN_NAME) or MODEL_STAGE not in {'sft', 'sft-smoke'}:
    raise ValueError('Choose a valid SFT run and stage')
from mmcultureqa.models.artifacts import require_merged_model
local_model = PROJECT_ROOT / 'outputs/inputs/models' / INPUT_RUN_NAME / MODEL_STAGE / 'best_model'
copy(cloud(f'task6-runs/{INPUT_RUN_NAME}/{MODEL_STAGE}/best_model'), local_model)
require_merged_model(local_model)
saved = json.loads((local_model / 'run_config.json').read_text(encoding='utf-8'))
predictions = PROJECT_ROOT / 'outputs/runs' / INPUT_RUN_NAME / 'predictions'
prediction_file = predictions / f'sft_{SPLIT}.jsonl'
overrides = {'model.name': str(local_model), 'model.revision': None, 'model.adapter': None,
             'model.adapter_revision': None, 'model.processor': str(local_model),
             'model.processor_revision': None, 'model.local_files_only': True, 'model.merged_model': True,
             'model.dtype': 'bfloat16', 'model.load_in_4bit': False,
             'inference.backend': 'vllm', 'inference.split': SPLIT,
             'inference.output': str(prediction_file), 'data.limit': TEST_LIMIT}
run_command('mmcultureqa.inference.predict', 'configs/inference.yaml', overrides)
if not prediction_file.is_file():
    raise FileNotFoundError(f'Inference produced no predictions: {prediction_file}')
upload_predictions(predictions, f'task6-runs/{INPUT_RUN_NAME}/predictions')


In [ ]:
from mmcultureqa.common import read_jsonl, write_jsonl
test_rows = read_jsonl(PROJECT_ROOT / 'data/prepared' / f'{SPLIT}.jsonl')
if TEST_LIMIT is not None:
    test_rows = test_rows[:TEST_LIMIT]
if test_rows and all(row.get('answer') for row in test_rows):
    gold = predictions / f'{SPLIT}_gold_en.jsonl'
    write_jsonl(gold, test_rows)
    stream_run([sys.executable, '-m', 'mmcultureqa.score', '--dtype', 'float16',
                '--gold', str(gold), '--lang', 'en', '--pred', str(prediction_file),
                '--out', str(predictions / f'sft_{SPLIT}_scores.json')], check=True)
    upload_predictions(predictions, f'task6-runs/{INPUT_RUN_NAME}/predictions')
else:
    print(f'{SPLIT} has no public answers. Predictions saved for submission; no local score available.')
